In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]    = "0"    # Kaggle cấp 2×T4; ép 1 GPU để giữ tính tái lập
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"    # hub v1.x: hf_transfer đã bị gỡ
print("OK")

OK


In [2]:
import glob, pathlib
for pat in ["wheels/*.whl", "llamacpp-*-static-*.tar.gz", "*.sha256",
            "base_manifest.json", "repo_id.txt"]:
    hits = glob.glob(f"/kaggle/working/{pat}")
    print(f"{pat:32s}", [pathlib.Path(h).name for h in hits] or "THIẾU")

wheels/*.whl                     ['llama_cpp_python-0.3.16-cp312-cp312-linux_x86_64.whl']
llamacpp-*-static-*.tar.gz       ['llamacpp-b10165-cpu-static-x64.tar.gz']
*.sha256                         ['wheel.sha256', 'llamacpp-b10165.sha256']
base_manifest.json               ['base_manifest.json']
repo_id.txt                      ['repo_id.txt']


In [3]:
!pip install -q "huggingface_hub==1.25.1" hf_xet
!pip install -q /kaggle/working/wheels/llama_cpp_python-*.whl
!pip install -q "transformers==5.5.0" "trl==0.24.0" "peft==0.20.0" "unsloth==2026.7.5" \
                accelerate bitsandbytes datasets sentencepiece protobuf safetensors

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 774.9/774.9 kB 11.6 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 63.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 7.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 71.1 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 24.1 MB/s eta 

In [4]:
!python -c "import torch, huggingface_hub as h, transformers as t, trl; \
print('torch          :', torch.__version__, '| cuda:', torch.cuda.is_available()); \
print('huggingface_hub:', h.__version__); \
print('transformers   :', t.__version__); \
print('trl            :', trl.__version__); \
assert torch.cuda.is_available(), 'torch MẤT CUDA'; \
assert not torch.__version__.endswith('+cpu'), 'torch bản CPU-only'; \
print('>>> CỔNG CHẶN: QUA')"

torch          : 2.10.0+cu128 | cuda: True
huggingface_hub: 1.25.1
transformers   : 5.5.0
trl            : 0.24.0
>>> CỔNG CHẶN: QUA


In [5]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, whoami

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
REPO = open("/kaggle/working/repo_id.txt").read().strip()
print("HF:", whoami()["name"], "| repo:", REPO)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF: dangnguyen254 | repo: dangnguyen254/thesis-graphrag-gguf


In [8]:
GH_USER   = "tandat-dao"        # <== SỬA thành username GitHub của bạn
GH_REPO   = "vn-legal-graphrag"
GH_BRANCH = "dev/fine-tune"
REPO_DIR  = f"/kaggle/working/{GH_REPO}"

import subprocess, pathlib

# Repo private: tạo Kaggle Secret tên GH_PAT (fine-grained, chỉ quyền Contents:Read)
try:
    from kaggle_secrets import UserSecretsClient
    _pat   = UserSecretsClient().get_secret("GH_PAT")
    GH_URL = f"https://{_pat}@github.com/{GH_USER}/{GH_REPO}.git"
    print("Dùng PAT (repo private)")
except Exception:
    GH_URL = f"https://github.com/{GH_USER}/{GH_REPO}.git"
    print("Dùng URL công khai")

assert "<" not in GH_URL and ">" not in GH_URL, "Còn placeholder trong GH_USER/GH_REPO"

def git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    print(out.replace(_pat, "***") if "_pat" in dir() else out)   # không lộ token
    r.check_returncode()

if pathlib.Path(REPO_DIR, ".git").exists():
    git("-C", REPO_DIR, "fetch", "--all", "--prune")
    git("-C", REPO_DIR, "checkout", GH_BRANCH)
    git("-C", REPO_DIR, "pull", "--ff-only")
else:
    git("clone", "--branch", GH_BRANCH, GH_URL, REPO_DIR)

r = subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--format=%H%n  %s%n  %ci"],
                   capture_output=True, text=True)
print("\n" + r.stdout)
print(">>> GHI COMMIT HASH NÀY VÀO KHÓA LUẬN — nó ghim phiên bản CODE")

Dùng URL công khai
Cloning into '/kaggle/working/vn-legal-graphrag'...

eecdc7b7dc533f2c564d74e795704fb4dcbb81a2
  feat(finetune)
  2026-07-29 12:53:53 +0700

>>> GHI COMMIT HASH NÀY VÀO KHÓA LUẬN — nó ghim phiên bản CODE


In [9]:
!echo "===================== finetune/README.md ====================="
!cat {REPO_DIR}/finetune/README.md
!echo ""
!echo "===================== các file phụ thuộc ====================="
!ls -la {REPO_DIR}
!for f in requirements.txt pyproject.toml finetune/requirements.txt; do \
   [ -f {REPO_DIR}/$f ] && echo "--- $f ---" && cat {REPO_DIR}/$f; done

===================== finetune/README.md =====================
# `finetune/` — Bổ sung mô hình sinh cục bộ vào Chương 4

Kế hoạch: [`docs/FINETUNE_EXECUTION_PLAN.md`](../docs/FINETUNE_EXECUTION_PLAN.md) (v2.3).

Nguyên tắc nền: **đóng băng truy hồi, chỉ đổi mô hình sinh.** Các file
`data/evaluation/results_*.json` đã lưu chuỗi `context` byte-identical với cái mô
hình sinh thực sự nhận → không cần Neo4j, không cần Qdrant, không gọi API.
Nếu ở bước nào thấy cần khởi động DB hoặc gọi Vertex AI thì **hướng đi đã sai**.

## Thư mục

| Đường dẫn | Nội dung |
|---|---|
| `replay.py` | Bộ phát lại (FT-02): đọc results JSON → dựng prompt → gọi mô hình → chấm bằng `src.evaluation.metrics` |
| `select_gate_ids.py` | Chọn 15 câu phân tầng cho gate FT-03 → `data/gate_ids.json` |
| `measure_token_budget.py` | FT-01A: đo ngân sách token thật |
| `recover_response_mode.py` | FT-01B: khôi phục `response_mode` → `data/mode_map.json` |
| `data/` | `mode_map.json`, `gate_ids.json` (`*.jsonl` bị gitignore)

In [11]:
import pathlib

pathlib.Path("/kaggle/working/constraints.txt").write_text("""\
torch==2.10.0
torchvision==0.25.0
torchaudio==2.10.0
transformers==5.5.0
trl==0.24.0
peft==0.20.0
unsloth==2026.7.5
huggingface_hub==1.25.1
llama_cpp_python==0.3.16
""")
print(open("/kaggle/working/constraints.txt").read())

torch==2.10.0
torchvision==0.25.0
torchaudio==2.10.0
transformers==5.5.0
trl==0.24.0
peft==0.20.0
unsloth==2026.7.5
huggingface_hub==1.25.1
llama_cpp_python==0.3.16



In [12]:
import pathlib
for f in ["requirements.txt", "finetune/requirements.txt", "pyproject.toml", "environment.yml"]:
    p = pathlib.Path(REPO_DIR, f)
    print(f"{'CÓ    ' if p.exists() else 'THIẾU '}{f}")
    if p.exists():
        print(p.read_text()[:1500])
        print("-" * 60)

CÓ    requirements.txt
neo4j>=5.0,<6.0
qdrant-client>=1.9.0
python-dotenv>=1.0.0
pytest>=8.0.0
sentence-transformers>=3.0.0
pyyaml>=6.0
anthropic>=0.40.0
rich>=13.7.0
google-genai>=1.0.0   # dự phòng Gemini khi Claude sập (demo) — import lazy, chỉ cần khi fallback nổ
numpy>=1.26            # expanded_eval: bootstrap CI (dep của sentence-transformers, khai tường minh vì import trực tiếp)
scipy>=1.11            # expanded_eval: Wilcoxon signed-rank
torch>=2.0             # reranker (D-20, teacher) — đã là dep của sentence-transformers
fastapi>=0.110         # UI demo bảo vệ (ui/server.py) — chỉ cần khi chạy web UI
uvicorn>=0.27          # ASGI server cho UI demo: uvicorn ui.server:app --port 8000

------------------------------------------------------------
THIẾU finetune/requirements.txt
THIẾU pyproject.toml
THIẾU environment.yml


In [13]:
import pathlib
req = next((p for f in ["requirements.txt", "finetune/requirements.txt"]
            if (p := pathlib.Path(REPO_DIR, f)).exists()), None)

if req:
    !pip install -q -r {req} -c /kaggle/working/constraints.txt
else:
    !pip install -q neo4j -c /kaggle/working/constraints.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.9/313.9 kB 5.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 35.6 MB/s eta 0:00:00


In [15]:
!python -c "import torch, huggingface_hub as h, transformers as t, trl; \
print('torch          :', torch.__version__, '| cuda:', torch.cuda.is_available()); \
print('huggingface_hub:', h.__version__); print('transformers   :', t.__version__); \
assert torch.cuda.is_available(), 'torch MẤT CUDA'; \
assert not torch.__version__.endswith('+cpu'), 'torch bản CPU-only'; \
print('>>> CỔNG CHẶN: QUA')"

torch          : 2.10.0+cu128 | cuda: True
huggingface_hub: 1.25.1
transformers   : 5.5.0
>>> CỔNG CHẶN: QUA


In [16]:
import subprocess, re
for _ in range(10):
    r = subprocess.run('python -c "from finetune.replay import build_chat_messages; print(\'OK\')"',
                       shell=True, cwd=REPO_DIR, capture_output=True, text=True)
    if "OK" in r.stdout:
        print("IMPORT OK"); break
    m = re.search(r"No module named '([^']+)'", r.stderr)
    if not m:
        print(r.stderr[-1500:]); break
    mod = m.group(1).split(".")[0]
    print("thiếu:", mod, "-> cài")
    subprocess.run(f"pip install -q {mod} -c /kaggle/working/constraints.txt", shell=True)
else:
    print("Quá 10 vòng — có gì đó không ổn")

IMPORT OK


In [17]:
!pip list 2>/dev/null | grep -iE '^(neo4j|qdrant|sentence-transformers|pydantic|networkx|scipy|numpy) '

neo4j                                    5.28.4
networkx                                 3.6.1
numpy                                    2.0.2
pydantic                                 2.12.3
scipy                                    1.16.3
sentence-transformers                    5.4.1


In [18]:
import subprocess
r = subprocess.run(
    'python -c "from finetune.replay import build_chat_messages; print(\'OK\')"',
    shell=True, cwd=REPO_DIR, capture_output=True, text=True)
print(r.stdout or r.stderr)

OK



In [19]:
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
import os; os.environ["PYTHONPATH"] = REPO_DIR
from finetune.replay import build_chat_messages
print("OK — import được từ trong kernel")

OK — import được từ trong kernel


In [23]:
from huggingface_hub import hf_hub_download
import hashlib, pathlib, json

MF  = pathlib.Path("/kaggle/working/base_manifest.json")
old = json.loads(MF.read_text())
p   = hf_hub_download(old["repo"], old["file"], revision=old["revision"])
h   = hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest()
assert h == old["sha256"], f"sha256 LỆCH!\n  chờ  {old['sha256']}\n  thấy {h}"

old["path"] = p
MF.write_text(json.dumps(old, indent=2))
print("OK, sha256 khớp bản đã ghi.")
print("path mới:", p)

INFO HTTP Request: HEAD https://huggingface.co/bartowski/Qwen_Qwen3-4B-Instruct-2507-GGUF/resolve/ae44f08e1392f39c0e474af10c3ff8355c8b6688/Qwen_Qwen3-4B-Instruct-2507-Q4_K_M.gguf "HTTP/1.1 302 Found"


Qwen_Qwen3-4B-Instruct-2507-Q4_K_M.gguf: reconstructing file:   0%|          |  0.00B / 2.50GB            

Qwen_Qwen3-4B-Instruct-2507-Q4_K_M.gguf: downloading bytes:           |  0.00B            

OK, sha256 khớp bản đã ghi.
path mới: /root/.cache/huggingface/hub/models--bartowski--Qwen_Qwen3-4B-Instruct-2507-GGUF/snapshots/ae44f08e1392f39c0e474af10c3ff8355c8b6688/Qwen_Qwen3-4B-Instruct-2507-Q4_K_M.gguf


In [24]:
import json, os, pathlib

m   = json.loads(pathlib.Path("/kaggle/working/base_manifest.json").read_text())
src = pathlib.Path(m["path"])
assert src.exists(), f"Đích không tồn tại: {src}\n>>> Chạy ô C2 để tải lại trước."

dst = pathlib.Path(REPO_DIR, GGUF_REL)
dst.parent.mkdir(parents=True, exist_ok=True)
if dst.exists() or dst.is_symlink():
    dst.unlink()
os.symlink(src, dst)
pathlib.Path(REPO_DIR, "finetune/results").mkdir(parents=True, exist_ok=True)

assert dst.stat().st_size == src.stat().st_size, "symlink gãy"
print("symlink OK :", dst)
print("        -> :", src)
print("kích thước :", round(dst.stat().st_size / 1e9, 2), "GB")

symlink OK : /kaggle/working/vn-legal-graphrag/finetune/models/Qwen3-4B-Instruct-2507-Q4_K_M.gguf
        -> : /root/.cache/huggingface/hub/models--bartowski--Qwen_Qwen3-4B-Instruct-2507-GGUF/snapshots/ae44f08e1392f39c0e474af10c3ff8355c8b6688/Qwen_Qwen3-4B-Instruct-2507-Q4_K_M.gguf
kích thước : 2.5 GB


In [25]:
!ls -laL {REPO_DIR}/finetune/models/
!head -c 4 {REPO_DIR}/finetune/models/Qwen3-4B-Instruct-2507-Q4_K_M.gguf | xxd

total 2438764
drwxr-xr-x 2 root root       4096 Jul 29 06:37 .
drwxr-xr-x 8 root root       4096 Jul 29 06:36 ..
-rw-r--r-- 1 root root 2497280736 Jul 29 06:37 Qwen3-4B-Instruct-2507-Q4_K_M.gguf
00000000: 4747 5546                                GGUF


In [26]:
import pathlib

GGUF_REL = "finetune/models/Qwen3-4B-Instruct-2507-Q4_K_M.gguf"
SRC_REL  = "data/evaluation/results_graphrag_20260710-085236.json"
IDS_REL  = "finetune/data/gate_ids.json"

for rel in [SRC_REL, IDS_REL, GGUF_REL]:
    p = pathlib.Path(REPO_DIR, rel)
    size = f"  ({p.stat().st_size/1e9:.2f} GB)" if p.exists() and p.stat().st_size > 1e8 else ""
    print(("CÓ    " if p.exists() else "THIẾU ") + rel + size)

CÓ    data/evaluation/results_graphrag_20260710-085236.json
CÓ    finetune/data/gate_ids.json
CÓ    finetune/models/Qwen3-4B-Instruct-2507-Q4_K_M.gguf  (2.50 GB)


In [27]:
!grep -n 'seed\|temperature\|top_p\|top_k\|min_p\|presence_penalty\|max_tokens\|n_ctx\|n_gpu_layers' \
    {REPO_DIR}/finetune/replay.py | head -30

131:      - `generation_config.json` của repo: `do_sample: true`, `temperature: 0.7`,
132:        `top_k: 20`, `top_p: 0.8`.
134:        TopK=20, and MinP=0" và "you can adjust the `presence_penalty` parameter
148:    > ở mục presence_penalty của chính card 2507 — không phải câu trích trên.
150:    ⚠️ **Đánh đổi tính tất định.** Kế hoạch §TASK-FT-02 ghi "temperature 0, seed cố
151:    định". Với `temperature > 0` thì tái lập chỉ còn ở mức *cùng seed + cùng build
153:    vẫn có phương sai giữa các seed → ảnh hưởng cách đọc N=1 ở FT-06.
159:    temperature: float = 0.7
160:    top_p: float = 0.8
161:    top_k: int = 20
162:    min_p: float = 0.0
166:    presence_penalty: float = 1.0
167:    seed: int = 42
172:            "temperature": self.temperature,
173:            "top_p": self.top_p,
174:            "top_k": self.top_k,
175:            "min_p": self.min_p,
176:            "presence_penalty": self.presence_penalty,
177:            "seed": self.seed,
179:            "greedy": self.te

In [28]:
!grep -n 'n_ctx\|n_gpu_layers\|n_batch\|add_argument' \
    {REPO_DIR}/finetune/replay.py | grep -iE 'default|add_argument|= *[0-9]'

541:                gp: GenParams, n_ctx: int = DEFAULT_N_CTX) -> dict:
622:    ap.add_argument("--input", required=True, type=Path,
624:    ap.add_argument("--model", required=True,
626:    ap.add_argument("--out", type=Path, default=None, help="file JSON đầu ra")
627:    ap.add_argument("--limit", type=int, default=0, help="chỉ chạy N câu đầu (0 = full)")
628:    ap.add_argument("--ids", default="",
631:    ap.add_argument("--n-shot", type=int, default=0, choices=[0, 1, 2],
633:    ap.add_argument("--resume", action="store_true", help="bỏ qua item đã có trong file .partial.jsonl")
634:    ap.add_argument("--seed", type=int, default=42)
635:    ap.add_argument("--max-new-tokens", type=int, default=MAX_NEW_TOKENS)
636:    ap.add_argument("--n-ctx", type=int, default=DEFAULT_N_CTX)
637:    ap.add_argument("--n-gpu-layers", type=int, default=-1, help="-1 = đẩy hết lên GPU")
638:    ap.add_argument("--tag", default="", help="nhãn thêm vào tên file đầu ra")
639:    ap.add_argument("--dump-

In [29]:
!sed -n '200,240p' {REPO_DIR}/finetune/replay.py

    name: str = "abstract"

    def generate(self, messages: list[dict], gp: GenParams) -> Generation:
        raise NotImplementedError

    def render_prompt(self, messages: list[dict]) -> tuple[str | None, str]:
        """Prompt ĐÃ RENDER qua chat template. Trả (chuỗi | None, tên_phương_pháp)."""
        return None, "khong-ho-tro"

    def count_prompt_tokens(self, messages: list[dict]) -> tuple[int | None, str]:
        """Số token của prompt đã render. Trả (số | None, tên_phương_pháp).

        None = backend không đo được → bỏ qua kiểm tràn n_ctx (có cảnh báo).
        """
        return None, "khong-ho-tro"


class MockBackend(Backend):
    """Trả chuỗi cố định. Dùng để verify toàn bộ đường đi schema mà không cần weights.

    `--model mock`            → chuỗi mặc định có 1 citation hợp lệ
    `--model mock:empty`      → chuỗi KHÔNG parse ra citation nào (test format_ok=False)
    `--model mock:cap`        → mô phỏng chạm trần token (test hit_token_cap)
    `--model mock:@path

In [32]:
import json, os, pathlib

os.environ["GGUF"] = GGUF_REL
os.environ["SRC"]  = SRC_REL

ids = json.load(open(pathlib.Path(REPO_DIR, IDS_REL), encoding="utf-8"))["ids_csv"]
os.environ["IDS"]  = ids

print("số câu :", len(ids.split(",")), "(kỳ vọng 15)")
print("IDS    :", ids[:100], "...")

số câu : 15 (kỳ vọng 15)
IDS    : V078,V082,V132,V001,V006,V021,V023,V026,V030,V040,V043,V042,V005,V105,V117 ...


In [33]:
!echo "GGUF=[$GGUF]"
!echo "SRC =[$SRC]"
!echo "IDS =[$IDS]" | cut -c1-90
!test -n "$SRC" -a -n "$IDS" -a -n "$GGUF" && echo ">>> BA BIẾN ĐỀU CÓ" || echo ">>> CÓ BIẾN RỖNG — DỪNG"

GGUF=[finetune/models/Qwen3-4B-Instruct-2507-Q4_K_M.gguf]
SRC =[data/evaluation/results_graphrag_20260710-085236.json]
IDS =[V078,V082,V132,V001,V006,V021,V023,V026,V030,V040,V043,V042,V005,V105,V117]
>>> BA BIẾN ĐỀU CÓ


In [34]:
%cd {REPO_DIR}

/kaggle/working/vn-legal-graphrag


In [35]:
!python -m finetune.replay --input "$SRC" --model mock --ids "$IDS" \
  --n-shot 0 --tag mock-s0 --dump-prompt finetune/results/prompt_mock-s0.txt

--ids: chọn 15/137 câu
backend=mock n_shot=0 n_ctx=16384 INCLUDE_SCHEMA_B=False
  tham số sinh: {"temperature": 0.7, "top_p": 0.8, "top_k": 20, "min_p": 0.0, "presence_penalty": 1.0, "seed": 42, "max_new_tokens": 2048, "greedy": false}
  --dump-prompt → finetune/results/prompt_mock-s0.txt (render=mock-noi-tho, None token)
  ⚠️  backend không đo được độ dài prompt (khong-ho-tro) → BỎ QUA kiểm tràn n_ctx. Chấp nhận được với mock; với weights thật thì không.
  [1/15] V001 mode=general cit=1 F1=1.00
  [2/15] V005 mode=irac    cit=1 F1=0.00
  [3/15] V006 mode=general cit=1 F1=0.00
  [4/15] V021 mode=general cit=1 F1=0.00
  [5/15] V023 mode=general cit=1 F1=0.00
  [6/15] V026 mode=general cit=1 F1=0.00
  [7/15] V030 mode=general cit=1 F1=0.00
  [8/15] V040 mode=general cit=1 F1=0.00
  [9/15] V042 mode=general cit=1 F1=0.00
  [10/15] V043 mode=general cit=1 F1=0.00
  [11/15] V078 mode=general cit=1 F1=0.00
  [12/15] V082 mode=general cit=1 F1=0.00
  [13/15] V105 mode=general cit=1 F1=0.00
  [

In [36]:
!python -m finetune.replay --input "$SRC" --model mock:empty --ids "$IDS" \
  --n-shot 0 --tag mock-empty

--ids: chọn 15/137 câu
backend=mock:empty n_shot=0 n_ctx=16384 INCLUDE_SCHEMA_B=False
  tham số sinh: {"temperature": 0.7, "top_p": 0.8, "top_k": 20, "min_p": 0.0, "presence_penalty": 1.0, "seed": 42, "max_new_tokens": 2048, "greedy": false}
  ⚠️  backend không đo được độ dài prompt (khong-ho-tro) → BỎ QUA kiểm tràn n_ctx. Chấp nhận được với mock; với weights thật thì không.
  [1/15] V001 mode=general cit=0 F1=0.00
  [2/15] V005 mode=irac    cit=0 F1=1.00
  [3/15] V006 mode=general cit=0 F1=0.00
  [4/15] V021 mode=general cit=0 F1=0.00
  [5/15] V023 mode=general cit=0 F1=0.00
  [6/15] V026 mode=general cit=0 F1=0.00
  [7/15] V030 mode=general cit=0 F1=0.00
  [8/15] V040 mode=general cit=0 F1=0.00
  [9/15] V042 mode=general cit=0 F1=0.00
  [10/15] V043 mode=general cit=0 F1=0.00
  [11/15] V078 mode=general cit=0 F1=0.00
  [12/15] V082 mode=general cit=0 F1=0.00
  [13/15] V105 mode=general cit=0 F1=1.00
  [14/15] V117 mode=general cit=0 F1=1.00
  [15/15] V132 mode=irac    cit=0 F1=0.00



In [37]:
!python -m finetune.replay --input "$SRC" --model mock:cap --ids "$IDS" \
  --n-shot 0 --tag mock-cap

--ids: chọn 15/137 câu
backend=mock:cap n_shot=0 n_ctx=16384 INCLUDE_SCHEMA_B=False
  tham số sinh: {"temperature": 0.7, "top_p": 0.8, "top_k": 20, "min_p": 0.0, "presence_penalty": 1.0, "seed": 42, "max_new_tokens": 2048, "greedy": false}
  ⚠️  backend không đo được độ dài prompt (khong-ho-tro) → BỎ QUA kiểm tràn n_ctx. Chấp nhận được với mock; với weights thật thì không.
  [1/15] V001 mode=general cit=0 F1=0.00 CAP!
  [2/15] V005 mode=irac    cit=0 F1=1.00 CAP!
  [3/15] V006 mode=general cit=0 F1=0.00 CAP!
  [4/15] V021 mode=general cit=0 F1=0.00 CAP!
  [5/15] V023 mode=general cit=0 F1=0.00 CAP!
  [6/15] V026 mode=general cit=0 F1=0.00 CAP!
  [7/15] V030 mode=general cit=0 F1=0.00 CAP!
  [8/15] V040 mode=general cit=0 F1=0.00 CAP!
  [9/15] V042 mode=general cit=0 F1=0.00 CAP!
  [10/15] V043 mode=general cit=0 F1=0.00 CAP!
  [11/15] V078 mode=general cit=0 F1=0.00 CAP!
  [12/15] V082 mode=general cit=0 F1=0.00 CAP!
  [13/15] V105 mode=general cit=0 F1=1.00 CAP!
  [14/15] V117 mode=ge

In [38]:
!python -m finetune.replay --input "$SRC" --model mock --ids "$IDS" \
  --n-shot 2 --tag mock-s2 --dump-prompt finetune/results/prompt_mock-s2.txt

--ids: chọn 15/137 câu
backend=mock n_shot=2 n_ctx=16384 INCLUDE_SCHEMA_B=False
  tham số sinh: {"temperature": 0.7, "top_p": 0.8, "top_k": 20, "min_p": 0.0, "presence_penalty": 1.0, "seed": 42, "max_new_tokens": 2048, "greedy": false}
  --dump-prompt → finetune/results/prompt_mock-s2.txt (render=mock-noi-tho, None token)
  ⚠️  backend không đo được độ dài prompt (khong-ho-tro) → BỎ QUA kiểm tràn n_ctx. Chấp nhận được với mock; với weights thật thì không.
  [1/15] V001 mode=general cit=1 F1=1.00
  [2/15] V005 mode=irac    cit=1 F1=0.00
  [3/15] V006 mode=general cit=1 F1=0.00
  [4/15] V021 mode=general cit=1 F1=0.00
  [5/15] V023 mode=general cit=1 F1=0.00
  [6/15] V026 mode=general cit=1 F1=0.00
  [7/15] V030 mode=general cit=1 F1=0.00
  [8/15] V040 mode=general cit=1 F1=0.00
  [9/15] V042 mode=general cit=1 F1=0.00
  [10/15] V043 mode=general cit=1 F1=0.00
  [11/15] V078 mode=general cit=1 F1=0.00
  [12/15] V082 mode=general cit=1 F1=0.00
  [13/15] V105 mode=general cit=1 F1=0.00
  [

In [39]:
!grep -n 'DEFAULT_N_CTX\|MAX_NEW_TOKENS' {REPO_DIR}/finetune/replay.py | head -5
!grep -n 'class .*Backend\|def count_prompt_tokens\|def render_prompt' {REPO_DIR}/finetune/replay.py

83:MAX_NEW_TOKENS = 2048  # §9.4(3). Gemini p50=257 max=1313 nhưng khối trích dẫn nằm
86:DEFAULT_N_CTX = 16384  # token_budget.md §2.7(1)
168:    max_new_tokens: int = MAX_NEW_TOKENS
541:                gp: GenParams, n_ctx: int = DEFAULT_N_CTX) -> dict:
635:    ap.add_argument("--max-new-tokens", type=int, default=MAX_NEW_TOKENS)
197:class Backend:
205:    def render_prompt(self, messages: list[dict]) -> tuple[str | None, str]:
209:    def count_prompt_tokens(self, messages: list[dict]) -> tuple[int | None, str]:
217:class MockBackend(Backend):
251:    def render_prompt(self, messages: list[dict]) -> tuple[str | None, str]:
256:    def count_prompt_tokens(self, messages: list[dict]) -> tuple[int | None, str]:
263:class LlamaCppBackend(Backend):
299:    def render_prompt(self, messages: list[dict]) -> tuple[str | None, str]:
342:    def count_prompt_tokens(self, messages: list[dict]) -> tuple[int | None, str]:


In [ ]:
from transformers import AutoTokenizer
import glob, pathlib
tk = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B-Instruct-2507")
for f in sorted(glob.glob(f"{REPO_DIR}/finetune/results/prompt_mock-*.txt")):
    n = len(tk(pathlib.Path(f).read_text(encoding="utf-8"), add_special_tokens=False)["input_ids"])
    print(f"{pathlib.Path(f).name:28s} {n:7,} token"
          + ("   <== VƯỢT 16384" if n > 16384 else ""))

In [ ]:
import json, os, pathlib

os.environ["GGUF"] = GGUF_REL
os.environ["SRC"]  = SRC_REL

ids = json.load(open(pathlib.Path(REPO_DIR, IDS_REL), encoding="utf-8"))["ids_csv"]
os.environ["IDS"]  = ids

print("số câu :", len(ids.split(",")), "(kỳ vọng 15)")
print("IDS    :", ids[:100], "...")

In [ ]:
%cd {REPO_DIR}

In [40]:
!python -m finetune.replay --input "$SRC" --model "$GGUF" --ids "$IDS" \
  --n-shot 0 --presence-penalty 1.0 --tag gate-s0-pp10 \
  --dump-prompt finetune/results/prompt_gate-s0-pp10.txt

--ids: chọn 15/137 câu
llama_context: n_ctx_per_seq (16384) < n_ctx_train (262144) -- the full capacity of the model will not be utilized
backend=llama-cpp:Qwen3-4B-Instruct-2507-Q4_K_M.gguf n_shot=0 n_ctx=16384 INCLUDE_SCHEMA_B=False
  tham số sinh: {"temperature": 0.7, "top_p": 0.8, "top_k": 20, "min_p": 0.0, "presence_penalty": 1.0, "seed": 42, "max_new_tokens": 2048, "greedy": false}
  --dump-prompt → finetune/results/prompt_gate-s0-pp10.txt (render=render-loi:UndefinedError, 10133 token)
  kiểm tràn n_ctx: BẬT (phương pháp xap-xi(render-loi:UndefinedError), ngưỡng 16384 - 2048 = 14336 token cho prompt)
  [1/15] V001 mode=general prompt=10133tok cit=0 F1=0.00
  [2/15] V005 mode=irac    prompt=9726tok cit=0 F1=1.00
  [3/15] V006 mode=general prompt=7596tok cit=0 F1=0.00
  [4/15] V021 mode=general prompt=10046tok cit=1 F1=1.00
  [5/15] V023 mode=general prompt=7882tok cit=0 F1=0.00
  [6/15] V026 mode=general prompt=7604tok cit=0 F1=0.00
  [7/15] V030 mode=general prompt=7891tok cit=0

In [56]:
%cd {REPO_DIR}

/kaggle/working/vn-legal-graphrag


In [64]:
!python -m finetune.replay --input "$SRC" --model "$GGUF" --ids "$IDS" \
  --n-shot 0 --presence-penalty 1.0 --tag gate-s0-pp10 \
  --dump-prompt finetune/results/prompt_gate-s0-pp10.txt

--ids: chọn 15/137 câu
llama_context: n_ctx_per_seq (16384) < n_ctx_train (262144) -- the full capacity of the model will not be utilized
backend=llama-cpp:Qwen3-4B-Instruct-2507-Q4_K_M.gguf n_shot=0 n_ctx=16384 INCLUDE_SCHEMA_B=False
  tham số sinh: {"temperature": 0.7, "top_p": 0.8, "top_k": 20, "min_p": 0.0, "presence_penalty": 1.0, "seed": 42, "max_new_tokens": 2048, "greedy": false}
  --dump-prompt → finetune/results/prompt_gate-s0-pp10.txt (render=gguf-chat-template-jinja2, 10144 token)
  kiểm tràn n_ctx: BẬT (phương pháp gguf-chat-template-jinja2, ngưỡng 16384 - 2048 = 14336 token cho prompt)
  [1/15] V001 mode=general prompt=10144tok cit=0 F1=0.00
  [2/15] V005 mode=irac    prompt=9737tok cit=0 F1=1.00
  [3/15] V006 mode=general prompt=7607tok cit=0 F1=0.00
  [4/15] V021 mode=general prompt=10057tok cit=1 F1=1.00
  [5/15] V023 mode=general prompt=7893tok cit=0 F1=0.00
  [6/15] V026 mode=general prompt=7615tok cit=0 F1=0.00
  [7/15] V030 mode=general prompt=7902tok cit=0 F1=0.00

In [65]:
!python -m finetune.replay --input "$SRC" --model "$GGUF" --ids "$IDS" \
  --n-shot 0 --presence-penalty 1.0 --tag gate-s0-pp10 \
  --dump-prompt finetune/results/prompt_gate-s0-pp10.txt

--ids: chọn 15/137 câu
llama_context: n_ctx_per_seq (16384) < n_ctx_train (262144) -- the full capacity of the model will not be utilized
backend=llama-cpp:Qwen3-4B-Instruct-2507-Q4_K_M.gguf n_shot=0 n_ctx=16384 INCLUDE_SCHEMA_B=False
  tham số sinh: {"temperature": 0.7, "top_p": 0.8, "top_k": 20, "min_p": 0.0, "presence_penalty": 1.0, "seed": 42, "max_new_tokens": 2048, "greedy": false}
  --dump-prompt → finetune/results/prompt_gate-s0-pp10.txt (render=gguf-chat-template-jinja2, 10144 token)
  kiểm tràn n_ctx: BẬT (phương pháp gguf-chat-template-jinja2, ngưỡng 16384 - 2048 = 14336 token cho prompt)
  [1/15] V001 mode=general prompt=10144tok cit=0 F1=0.00
  [2/15] V005 mode=irac    prompt=9737tok cit=0 F1=1.00
  [3/15] V006 mode=general prompt=7607tok cit=0 F1=0.00
  [4/15] V021 mode=general prompt=10057tok cit=1 F1=1.00
  [5/15] V023 mode=general prompt=7893tok cit=0 F1=0.00
  [6/15] V026 mode=general prompt=7615tok cit=0 F1=0.00
  [7/15] V030 mode=general prompt=7902tok cit=0 F1=0.00

In [77]:
%cd {REPO_DIR}

/kaggle/working/vn-legal-graphrag


In [78]:
!python -m finetune.replay --input "$SRC" --model "$GGUF" --ids "$IDS" \
  --n-shot 2 --presence-penalty 1.0 --tag gate-s2-pp10 \
  --dump-prompt finetune/results/prompt_gate-s2-pp10.txt

--ids: chọn 15/137 câu
llama_context: n_ctx_per_seq (16384) < n_ctx_train (262144) -- the full capacity of the model will not be utilized
backend=llama-cpp:Qwen3-4B-Instruct-2507-Q4_K_M.gguf n_shot=2 n_ctx=16384 INCLUDE_SCHEMA_B=False
  tham số sinh: {"temperature": 0.7, "top_p": 0.8, "top_k": 20, "min_p": 0.0, "presence_penalty": 1.0, "seed": 42, "max_new_tokens": 2048, "greedy": false}
  --dump-prompt → finetune/results/prompt_gate-s2-pp10.txt (render=gguf-chat-template-jinja2, 10556 token)
  kiểm tràn n_ctx: BẬT (phương pháp gguf-chat-template-jinja2, ngưỡng 16384 - 2048 = 14336 token cho prompt)
  [1/15] V001 mode=general prompt=10556tok cit=1 F1=1.00
  [2/15] V005 mode=irac    prompt=10149tok cit=4 F1=0.00
  [3/15] V006 mode=general prompt=8019tok cit=0 F1=0.00
  [4/15] V021 mode=general prompt=10469tok cit=1 F1=1.00
  [5/15] V023 mode=general prompt=8305tok cit=2 F1=0.67
  [6/15] V026 mode=general prompt=8027tok cit=1 F1=0.00
  [7/15] V030 mode=general prompt=8314tok cit=3 F1=0.8

In [79]:
!python -m finetune.replay --input "$SRC" --model "$GGUF" --ids "$IDS" \
  --n-shot 2 --presence-penalty 0 --tag gate-s2-pp00

--ids: chọn 15/137 câu
llama_context: n_ctx_per_seq (16384) < n_ctx_train (262144) -- the full capacity of the model will not be utilized
backend=llama-cpp:Qwen3-4B-Instruct-2507-Q4_K_M.gguf n_shot=2 n_ctx=16384 INCLUDE_SCHEMA_B=False
  tham số sinh: {"temperature": 0.7, "top_p": 0.8, "top_k": 20, "min_p": 0.0, "presence_penalty": 0.0, "seed": 42, "max_new_tokens": 2048, "greedy": false}
  kiểm tràn n_ctx: BẬT (phương pháp gguf-chat-template-jinja2, ngưỡng 16384 - 2048 = 14336 token cho prompt)
  [1/15] V001 mode=general prompt=10556tok cit=1 F1=1.00
  [2/15] V005 mode=irac    prompt=10149tok cit=4 F1=0.00
  [3/15] V006 mode=general prompt=8019tok cit=0 F1=0.00
  [4/15] V021 mode=general prompt=10469tok cit=1 F1=1.00
  [5/15] V023 mode=general prompt=8305tok cit=2 F1=0.67
  [6/15] V026 mode=general prompt=8027tok cit=1 F1=0.00
  [7/15] V030 mode=general prompt=8314tok cit=4 F1=0.67
  [8/15] V040 mode=general prompt=8494tok cit=1 F1=1.00
  [9/15] V042 mode=general prompt=8426tok cit=1 F

In [76]:
!python -m finetune.replay --input "$SRC" --model "$GGUF" --ids "$IDS" \
  --n-shot 0 --presence-penalty 0 --tag gate-s0-pp00

--ids: chọn 15/137 câu
llama_context: n_ctx_per_seq (16384) < n_ctx_train (262144) -- the full capacity of the model will not be utilized
backend=llama-cpp:Qwen3-4B-Instruct-2507-Q4_K_M.gguf n_shot=0 n_ctx=16384 INCLUDE_SCHEMA_B=False
  tham số sinh: {"temperature": 0.7, "top_p": 0.8, "top_k": 20, "min_p": 0.0, "presence_penalty": 0.0, "seed": 42, "max_new_tokens": 2048, "greedy": false}
  kiểm tràn n_ctx: BẬT (phương pháp gguf-chat-template-jinja2, ngưỡng 16384 - 2048 = 14336 token cho prompt)
^C


In [80]:
import json, glob
for tag in ["gate-s0-pp10", "gate-s0-pp00", "gate-s2-pp10", "gate-s2-pp00"]:
    f = sorted(glob.glob(f"{REPO_DIR}/finetune/results/results_*{tag}_2026*.json"))[-1]
    d = json.load(open(f, encoding="utf-8"))["results"]
    lech = [r["prompt_len_lech"] for r in d if r.get("prompt_len_lech") is not None]
    mx   = max(r["n_tokens_prompt"] for r in d)
    print(f"{tag:16s} lệch {min(lech)}–{max(lech)}  prompt max={mx:,}  "
          f"{'VƯỢT 14336' if mx > 14336 else 'trong ngưỡng'}")

gate-s0-pp10     lệch 22–22  prompt max=11,211  trong ngưỡng
gate-s0-pp00     lệch 22–22  prompt max=11,211  trong ngưỡng
gate-s2-pp10     lệch 56–56  prompt max=11,623  trong ngưỡng
gate-s2-pp00     lệch 56–56  prompt max=11,623  trong ngưỡng


In [81]:
a, b = sorted(glob.glob(f"{REPO_DIR}/finetune/results/results_*gate-s0-pp10_2026*.json"))[-2:]
A = {r["id"]: r["answer"] for r in json.load(open(a, encoding="utf-8"))["results"]}
B = {r["id"]: r["answer"] for r in json.load(open(b, encoding="utf-8"))["results"]}
diff = [k for k in A if A[k] != B[k]]
print("khác:", len(diff), "/", len(A), diff or "→ TRÙNG KHÍT TỪNG KÝ TỰ")

khác: 0 / 15 → TRÙNG KHÍT TỪNG KÝ TỰ


In [82]:
HF_REPO = open("/kaggle/working/repo_id.txt").read().strip()
from huggingface_hub import upload_folder
upload_folder(folder_path=f"{REPO_DIR}/finetune/results",
              path_in_repo="session1_gate", repo_id=HF_REPO,
              allow_patterns=["*.json", "*.txt"])
print("đã lưu ->", HF_REPO)

INFO HTTP Request: POST https://huggingface.co/api/models/dangnguyen254/thesis-graphrag-gguf/preupload/main "HTTP/1.1 200 OK"
INFO HTTP Request: POST https://huggingface.co/api/models/dangnguyen254/thesis-graphrag-gguf/commit/main "HTTP/1.1 200 OK"


đã lưu -> dangnguyen254/thesis-graphrag-gguf
